# CIS Windows 11 RAG — Ingest PDF and Build a Persistent Chroma Vector Store

This notebook prepares the benchmark knowledge base before we build the RAG architectures.

## What this notebook does

1. Locate the CIS Windows 11 Benchmark PDF.
2. Load the PDF with LangChain's `PyPDFLoader`.
3. Inspect extracted pages and metadata.
4. Split the pages with `RecursiveCharacterTextSplitter`.
5. Create embeddings with LangChain's `HuggingFaceEmbeddings`.
6. Build and persist a Chroma vector store.
7. Run a small similarity-search smoke test.

**Design choice:** We use Chroma for this first persistent index because it supports a local on-disk store and integrates directly with LangChain. We can add FAISS as a comparison later.

**Important:** This notebook creates a retrieval index, not a compliance scanner. To assess a machine's actual compliance, we would also need trustworthy configuration or audit evidence.

## 1. Expected project layout

Run this notebook from the `notebooks/` folder in the agreed project.

```text
cis-windows11-rag/
├── data/
│   └── benchmark/
│       └── <your CIS Windows 11 Benchmark PDF>
├── notebooks/
│   ├── 00_environment_setup.ipynb
│   └── 00b_ingestion_vector_store.ipynb
├── vector_store/
│   └── chroma_cis_windows11/
├── evaluation_results/
├── requirements.txt
└── README.md
```

Place the PDF in `data/benchmark/`. The script deliberately does not assume a specific PDF filename.

In [8]:
from pathlib import Path
import sys
import hashlib
import json

# Resolve the project root from the notebook's current working directory.
# Recommended: open Jupyter with the project root as the working directory.
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").is_dir():
    PROJECT_ROOT = CURRENT_DIR
else:
    raise RuntimeError(
        "Cannot identify the project root. Open this notebook from the "
        "'cis-windows11-rag/notebooks' folder or start Jupyter in the project root."
    )

BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"

# Create required folders if they do not already exist.
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root     : {PROJECT_ROOT}")
print(f"Benchmark folder : {BENCHMARK_DIR}")
print(f"Vector store path: {VECTOR_STORE_DIR}")
print(f"Python           : {sys.version.split()[0]}")

Project root     : C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag
Benchmark folder : C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\data\benchmark
Vector store path: C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\vector_store\chroma_cis_windows11
Python           : 3.12.10


## 2. Configure ingestion

Adjust these values if needed. `CHUNK_SIZE` and `CHUNK_OVERLAP` are starting points, not universal best values. We will tune them later using retrieval tests.

**Rebuild behavior:** If a persistent store already exists, Chroma may contain earlier chunks. To avoid accidentally mixing old and new index contents, this notebook defaults to stopping when the target store already exists. Set `REBUILD_VECTOR_STORE = True` only when you intentionally want to remove and rebuild this workshop index.

In [9]:
# PDF and indexing settings
PDF_GLOB = "*.pdf"

# Chunk size is measured in characters by RecursiveCharacterTextSplitter.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

# Local sentence-transformers embedding model.
# The first run may download model files from the model hub.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Chroma persistence settings
COLLECTION_NAME = "cis_windows11_benchmark"

# Safety switch: False prevents accidental deletion of an existing index.
REBUILD_VECTOR_STORE = False

# When True, show a short preview of extracted text and chunk content.
SHOW_TEXT_PREVIEWS = True

print("Configuration loaded.")

Configuration loaded.


## 3. Find the benchmark PDF

If several PDFs are present, choose the intended file explicitly in the cell below. This avoids indexing the wrong version by accident.

In [10]:
pdf_candidates = sorted(BENCHMARK_DIR.glob(PDF_GLOB))

if not pdf_candidates:
    raise FileNotFoundError(
        f"No PDF files found in {BENCHMARK_DIR}. "
        "Copy the authorized CIS Windows 11 Benchmark PDF into this folder, then rerun this cell."
    )

print("PDF files found:")
for index, path in enumerate(pdf_candidates):
    print(f"  [{index}] {path.name} ({path.stat().st_size:,} bytes)")

# Default to the first PDF. If multiple files exist, change this index to select the right version.
PDF_SELECTION_INDEX = 0

if PDF_SELECTION_INDEX < 0 or PDF_SELECTION_INDEX >= len(pdf_candidates):
    raise IndexError("PDF_SELECTION_INDEX is outside the list of PDF files.")

PDF_PATH = pdf_candidates[PDF_SELECTION_INDEX]
PDF_SHA256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()

print(f"\nSelected PDF: {PDF_PATH.name}")
print(f"SHA-256     : {PDF_SHA256}")

PDF files found:
  [0] CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf (8,065,258 bytes)

Selected PDF: CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf
SHA-256     : c57ab2dd7dd130e379ec3d9f1febcf9751257e25c1a3472179394988e637c1f4


## 4. Load the PDF with LangChain

`PyPDFLoader` creates LangChain `Document` objects, generally one per PDF page. We preserve the page metadata because later answers should be able to point back to source pages.

PDF extraction quality matters: scanned pages may need OCR, and complex tables may not extract perfectly. We will inspect the output before indexing.

In [11]:
from langchain_community.document_loaders import PyPDFLoader

# Load pages as LangChain Document objects.
pdf_loader = PyPDFLoader(str(PDF_PATH))
pages = pdf_loader.load()

if not pages:
    raise ValueError(
        "The PDF loader returned no pages. Check that the PDF is valid and text-extractable."
    )

# Attach stable source metadata to every page. Page numbering from PyPDFLoader is usually zero-based,
# so we add a human-friendly one-based page number without discarding the original metadata.
for page_document in pages:
    page_document.metadata["source_file"] = PDF_PATH.name
    page_document.metadata["source_sha256"] = PDF_SHA256
    if "page" in page_document.metadata:
        page_document.metadata["page_number"] = int(page_document.metadata["page"]) + 1

print(f"Loaded {len(pages)} page documents.")

if SHOW_TEXT_PREVIEWS:
    first_page = pages[0]
    print("\nFirst page metadata:")
    print(first_page.metadata)
    print("\nFirst page text preview:")
    print(first_page.page_content[:1200])

Loaded 1454 page documents.

First page metadata:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-23T12:32:42-04:00', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_enabled': 'true', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_setdate': '2026-01-21T15:19:28Z', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_method': 'Privileged', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_name': 'External - Non-Public', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_siteid': '0db8894b-5a7a-4432-8cdc-399a3807bda0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_actionid': '87d580ef-3cdd-442b-9b12-72fea4086cd0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_contentbits': '0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_tag': '10, 0, 1, 1', 'author': 'CISUser', 'moddate': '2026-03-23T12:32:42-04:00', 'source': 'C:\\Users\\mindf\\OneDrive\\Projects\\mindful-learning-academy\\sapient-ai\\classwork\\ps2026-ai

## 5. Check extraction quality

This is a quick diagnostic, not a full PDF quality assessment. A large number of empty pages may indicate a scanned PDF or extraction problem. We should not silently index empty pages.

In [12]:
empty_pages = [
    index for index, document in enumerate(pages)
    if not document.page_content or not document.page_content.strip()
]

total_characters = sum(len(document.page_content) for document in pages)
nonempty_pages = len(pages) - len(empty_pages)

print(f"Total pages          : {len(pages)}")
print(f"Non-empty pages      : {nonempty_pages}")
print(f"Empty/blank pages    : {len(empty_pages)}")
print(f"Extracted characters : {total_characters:,}")

if empty_pages:
    print("Blank page indexes (zero-based):", empty_pages[:30])
    print("If important content is missing, use OCR or a layout-aware PDF extraction tool before proceeding.")

# Remove truly empty pages to avoid creating useless chunks.
pages = [
    document for document in pages
    if document.page_content and document.page_content.strip()
]

if not pages:
    raise ValueError(
        "No extractable text was found in the PDF. OCR or another extraction method is required."
    )

Total pages          : 1454
Non-empty pages      : 1454
Empty/blank pages    : 0
Extracted characters : 1,850,386


## 6. Split pages into chunks

We use `RecursiveCharacterTextSplitter` from LangChain. It attempts to split on sensible boundaries while keeping chunks within the target size.

The overlap retains some surrounding context between adjacent chunks. We preserve source page metadata so retrieved chunks can be traced to the benchmark.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,  # Records each chunk's character offset within its source page.
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(pages)

if not chunks:
    raise ValueError("Text splitting produced no chunks.")

# Add a stable identifier to each chunk for traceability and duplicate checks.
# This ID is metadata; Chroma will also maintain its own internal record IDs.
for chunk_number, chunk in enumerate(chunks):
    source_name = chunk.metadata.get("source_file", PDF_PATH.name)
    page_number = chunk.metadata.get("page_number", "unknown")
    start_index = chunk.metadata.get("start_index", -1)

    identity_text = (
        f"{PDF_SHA256}|{source_name}|{page_number}|{start_index}|{chunk.page_content}"
    )
    chunk.metadata["chunk_id"] = hashlib.sha256(
        identity_text.encode("utf-8")
    ).hexdigest()

print(f"Created {len(chunks)} chunks.")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.1f} characters")

if SHOW_TEXT_PREVIEWS:
    print("\nExample chunk metadata:")
    print(chunks[0].metadata)
    print("\nExample chunk text:")
    print(chunks[0].page_content[:1000])

Created 2638 chunks.
Average chunk length: 749.5 characters

Example chunk metadata:
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-23T12:32:42-04:00', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_enabled': 'true', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_setdate': '2026-01-21T15:19:28Z', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_method': 'Privileged', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_name': 'External - Non-Public', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_siteid': '0db8894b-5a7a-4432-8cdc-399a3807bda0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_actionid': '87d580ef-3cdd-442b-9b12-72fea4086cd0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_contentbits': '0', 'msip_label_6072e0d1-5c8d-43bf-81e4-f636eb0b0152_tag': '10, 0, 1, 1', 'author': 'CISUser', 'moddate': '2026-03-23T12:32:42-04:00', 'source': 'C:\\Users\\mindf\\OneDrive\\Projects\\mindful-learning-academ

## 7. Initialize the embedding model

The embedding model converts each chunk into a numeric vector. At query time, the same model converts the user's question into a vector so similar content can be retrieved.

This is a local Sentence Transformers model wrapped by LangChain's `HuggingFaceEmbeddings`. Its first run may download model files. If your environment must be fully offline, cache the model in advance.

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},  # Use "cuda" only if your setup supports it.
    encode_kwargs={"normalize_embeddings": True},
)

# Test the embedding model before building the full index.
test_vector = embeddings.embed_query(
    "Which Windows security policy does the CIS Benchmark recommend?"
)

if not test_vector:
    raise RuntimeError("The embedding model returned an empty vector.")

print(f"Embedding model : {EMBEDDING_MODEL_NAME}")
print(f"Vector dimension: {len(test_vector)}")
print(f"Vector preview  : {test_vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model : sentence-transformers/all-MiniLM-L6-v2
Vector dimension: 384
Vector preview  : [-0.04406122863292694, 0.033494655042886734, -0.05349390208721161, 0.007189556024968624, 0.01245950348675251]


## 8. Build and persist the Chroma vector store

This cell writes the chunks and their embeddings to disk. The rebuild guard is intentionally conservative: it refuses to overwrite a non-empty target unless `REBUILD_VECTOR_STORE` is explicitly enabled.

**Caution:** Rebuilding deletes the selected workshop index directory. Confirm that it contains only this project's generated index before enabling the rebuild option.

In [15]:
import shutil
from langchain_chroma import Chroma

# Guard against accidental duplicate ingestion or mixing indexes from different PDF versions.
if VECTOR_STORE_DIR.exists() and any(VECTOR_STORE_DIR.iterdir()):
    if not REBUILD_VECTOR_STORE:
        raise FileExistsError(
            f"The target vector-store directory is not empty: {VECTOR_STORE_DIR}\n"
            "To intentionally rebuild it, set REBUILD_VECTOR_STORE = True and rerun this cell. "
            "This removes the existing contents of this specific workshop index directory."
        )

    # Remove only the dedicated Chroma index directory configured above.
    shutil.rmtree(VECTOR_STORE_DIR)

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

# Add chunks with deterministic IDs so each source chunk has a traceable identifier.
chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=chunk_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
)

print("Chroma vector store created.")
print(f"Persisted directory: {VECTOR_STORE_DIR}")
print(f"Collection name    : {COLLECTION_NAME}")
print(f"Chunks indexed     : {len(chunks)}")

Chroma vector store created.
Persisted directory: C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\vector_store\chroma_cis_windows11
Collection name    : cis_windows11_benchmark
Chunks indexed     : 2638


## 9. Verify persistence by reopening the store

We reopen Chroma from its directory rather than relying only on the in-memory object. This helps verify that the persisted index can be used by later notebooks.

In [16]:
# Reopen the persisted vector store with the same embedding model and collection name.
reopened_vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(VECTOR_STORE_DIR),
)

# Chroma exposes the underlying collection count through its collection object.
indexed_count = reopened_vector_store._collection.count()

print(f"Persisted collection count: {indexed_count}")

if indexed_count != len(chunks):
    raise RuntimeError(
        f"Expected {len(chunks)} indexed chunks but found {indexed_count}. "
        "Review the index build and persistence configuration."
    )

print("Persistence check passed.")

Persisted collection count: 2638
Persistence check passed.


## 10. Run a similarity-search smoke test

Choose a question relevant to your benchmark version. This is only a retrieval test: it does not call a chat model or generate an answer yet.

In [17]:
# Replace this sample with a question you want the index to answer.
TEST_QUERY = "What does the benchmark say about Windows Defender Firewall configuration?"
TOP_K = 4

retrieved_documents = reopened_vector_store.similarity_search(
    TEST_QUERY,
    k=TOP_K,
)

print(f"Query: {TEST_QUERY}")
print(f"Retrieved {len(retrieved_documents)} documents.\n")

for rank, document in enumerate(retrieved_documents, start=1):
    page_number = document.metadata.get("page_number", "unknown")
    recommendation_id = document.metadata.get("recommendation_id", "not extracted")
    print("=" * 80)
    print(f"Rank: {rank} | Page: {page_number} | Recommendation ID: {recommendation_id}")
    print(f"Source: {document.metadata.get('source_file', 'unknown')}")
    print(document.page_content[:1200])

Query: What does the benchmark say about Windows Defender Firewall configuration?
Retrieved 4 documents.

Rank: 1 | Page: 373 | Recommendation ID: not extracted
Source: CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf
Page 372 
6 Registry 
This section is intentionally blank and exists to ensure the structure of Windows 
benchmarks is consistent. 
7 File System 
This section is intentionally blank and exists to ensure the structure of Windows 
benchmarks is consistent. 
8 Wired Network (IEEE 802.3) Policies 
This section is intentionally blank and exists to ensure the structure of Windows 
benchmarks is consistent. 
9 Windows Defender Firewall with Advanced Security 
(formerly Windows Firewall with Advanced Security) 
This section contains recommendations for configuring the Windows Firewall. 
Note: In older versions of Microsoft Windows, this section was named Windows Firewall 
with Advanced Security, but it was renamed to Windows Defender Firewall with 
Advanced Security star

## 11. Save an ingestion manifest

The manifest records which source file and settings were used to build the index. This helps us reproduce experiments and detect when the source PDF changes.

It does not contain the document text or embedding vectors.

In [18]:
from datetime import datetime, timezone

manifest = {
    "source_file": PDF_PATH.name,
    "source_sha256": PDF_SHA256,
    "page_count_indexed": len(pages),
    "chunk_count": len(chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "vector_store": "Chroma",
    "collection_name": COLLECTION_NAME,
    "persist_directory": str(VECTOR_STORE_DIR.relative_to(PROJECT_ROOT)),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = PROJECT_ROOT / "vector_store" / "ingestion_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Manifest saved to: {manifest_path}")
print(json.dumps(manifest, indent=2))

Manifest saved to: C:\Users\mindf\OneDrive\Projects\mindful-learning-academy\sapient-ai\classwork\ps2026-ai\retrieval-augmented-generation\cis-windows11-rag\vector_store\ingestion_manifest.json
{
  "source_file": "CIS_Microsoft_Windows_11_Enterprise_Benchmark_v5.0.1.pdf",
  "source_sha256": "c57ab2dd7dd130e379ec3d9f1febcf9751257e25c1a3472179394988e637c1f4",
  "page_count_indexed": 1454,
  "chunk_count": 2638,
  "chunk_size": 1000,
  "chunk_overlap": 150,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "vector_store": "Chroma",
  "collection_name": "cis_windows11_benchmark",
  "persist_directory": "vector_store\\chroma_cis_windows11",
  "created_at_utc": "2026-10-29T11:31:46.443939+00:00"
}


## 12. Completion checklist

- [ ] The correct CIS Windows 11 Benchmark PDF was selected.
- [ ] Extracted page text was inspected for quality.
- [ ] Chunks retain source filename and page metadata.
- [ ] The embedding model loaded and returned a vector.
- [ ] Chroma index was built and persisted.
- [ ] The persisted collection was reopened successfully.
- [ ] Similarity search returned plausible benchmark passages.
- [ ] The ingestion manifest was saved.

### What comes next?

In `01_naive_rag.ipynb`, we will reopen this vector store, retrieve relevant chunks for a user question, pass them to the chat model through a LangChain prompt and retrieval chain, and produce an answer with source references.

### Troubleshooting notes

- **No PDF found:** Copy the PDF into `data/benchmark/`.
- **Import error:** Install the requirements and restart the Jupyter kernel.
- **No text extracted:** The PDF may be scanned; OCR is needed before indexing.
- **Poor retrieval:** Check extraction, chunk size, overlap, and the embedding model; later we will compare vector, keyword, and hybrid search.
- **Existing index guard:** Set `REBUILD_VECTOR_STORE = True` only when intentionally rebuilding this dedicated index directory.